# LP arb tick replay — Static Arb (Method 1.2)

Replays a `live_tick_logs/tsmc_ticks_YYYYMMDD.csv` recording (from the Live
Arb tab's Record button) and, after every tick, reconstructs the current
warrant/option book and re-runs the **same** static-arbitrage LP the
`/match_static_arb` route runs — `logic/static_arb.py`'s `_scan_chain`,
called directly rather than through `match_static_arb` itself (which fetches
its own data live from Supabase and can't be replayed from a CSV). This is
the multi-leg "basket" method from Arb Method 1.2 in the pitch — the one that
pools warrants across several strikes to hedge an option's full 2,000-share
exercise ratio, instead of requiring one warrant to do it alone like Direct
Match.

**Cost note:** this notebook solves one LP *per option expiry currently in
the book*, after **every single tick** (`STRIDE = 1`). On the Rust engine
that is ~7ms per tick for a whole several-hundred-instrument chain, so a
100k-tick recording is a matter of minutes; on the scipy fallback it is ~70x
that and a long recording will not finish in one sitting. Either way nothing
is skipped, so a "no arb" result means no arb existed at any tick, not "we
didn't happen to check then." If a first pass is still too slow, raise
`STRIDE` in the run cell below (or narrow `MAX_HORIZON_DTE`) to trade
completeness for speed.

**Known divergences from the live route** (read before trusting a "no arb"
result):
- `match_static_arb` also filters options by `min_volume` (cumulative traded
  volume). The tick log only carries book depth (`bid_size`/`ask_size`), not
  traded volume, so that filter is skipped here — a thin-but-quoted book can
  surface a structure the live route would have dropped.
- Same `exercise_ratio` schema gap as the Direct Match notebook: a recording
  made before that CSV column existed will have every warrant leg silently
  excluded (the leg builder requires `ratio > 0`). The load cell below warns
  if it finds this.
- Contract size `M` is hardcoded to 2,000 shares/contract — correct for
  every TW single-stock option, but this notebook is scoped to single-stock
  underlyings like TSMC, not TXO (50 NT$/point).
- The Rust and scipy engines can pick different vertices of a tied optimal
  face, so on roughly 1% of horizons they report a different (equally valid,
  never less profitable) leg combination at the same LP optimum — see
  `tests/logic/test_static_arb_lp_parity.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Repo root on sys.path so `logic.static_arb` imports exactly as it does for
# app.py::match_static_arb — this notebook lives in notebooks/, so the repo
# root is one directory up.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from logic import iv_engine, static_arb, options_logic

# The LP runs in Rust when the extension is built (`./scripts/build_rust.sh`):
# ~7ms per tick over a whole chain against ~500ms for the scipy path, which is
# the difference between a replay that finishes and one that does not. Without
# it the notebook still runs, on scipy, at the old speed.
USE_RUST = iv_engine.SCAN_STATIC_ARB and iv_engine.use_rust("arb")
print("static_arb engine:", "rust" if USE_RUST else "python (scipy/HiGHS)")
print("risk-free rate R =", options_logic.R)

## Load the tick log

Point `CSV_PATH` at the file you downloaded (or leave it to auto-pick the
newest file under `live_tick_logs/`).

In [ ]:
# Set this explicitly to analyze a specific downloaded file, e.g.:
# CSV_PATH = Path("/path/to/tsmc_ticks_20260903.csv")
CSV_PATH = None

if CSV_PATH is None:
    candidates = sorted((REPO_ROOT / "live_tick_logs").glob("tsmc_ticks_*.csv"))
    if not candidates:
        raise FileNotFoundError("No tsmc_ticks_*.csv found under live_tick_logs/ — record a session first, "
                                 "or set CSV_PATH to a downloaded file.")
    CSV_PATH = candidates[-1]

print("Loading:", CSV_PATH)
raw = pd.read_csv(CSV_PATH, parse_dates=["ts"])
raw["expiry"] = pd.to_datetime(raw["expiry"], errors="coerce").dt.date
raw = raw.sort_values("ts", kind="stable").reset_index(drop=True)
print(f"{len(raw):,} ticks — {raw['kind'].value_counts().to_dict()}")

missing_ratio = raw[(raw["kind"] == "warrant") & raw["exercise_ratio"].isna()]
if len(missing_ratio):
    pct = len(missing_ratio) / (raw["kind"] == "warrant").sum() * 100
    print(f"WARNING: {len(missing_ratio):,} warrant ticks ({pct:.0f}%) have no exercise_ratio "
          f"— these will be silently excluded from every match below. "
          f"See the schema-gap note above.")

if not (raw["kind"] == "option").any():
    print("WARNING: this recording has zero option ticks — the LP's short side is "
          "options-only (warrants are BUY-ONLY), so no structure can ever be found "
          "without at least one option quote in the book.")

raw.head()

## Replay: fold each tick into a book cache, then re-solve the LP per horizon

`warrant_books`/`option_books` mirror the same in-memory "current book"
pattern as the Direct Match notebook — each tick fully replaces that one
code's cached row (both bid and ask sides). After folding each tick (every
one, at `STRIDE = 1`), the book is flattened into `warrant_df`/`opt_df`
DataFrames carrying exactly the columns the leg builder expects, restricted
to options with `days_to_expiry >= 1` (matching
`options_logic.read_tw_option`'s own `min_days=1`, which the live route
relies on upstream). `static_arb._scan_chain` then normalises the legs and
solves one LP per option-expiry horizon — the same work `match_static_arb`
does, just reading from the replayed book instead of a live fetch.

A horizon only produces a row when the structure it finds is still profitable
*after* rounding every leg to whole board lots (張) / contracts (口) — this
is the step from the pitch where "90% of trades get killed." A tick with no
viable horizon contributes no rows, same convention as the Direct Match
notebook.

In [ ]:
def _clean(v):
    """NaN/NaT -> None; everything else passed through unchanged."""
    if v is None or v is pd.NaT:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    return v


def _fold_warrant_tick(row):
    return {
        "code": row.code,
        "name": _clean(row.name) or row.code,
        "type": _clean(row.type),
        "strike": _clean(row.strike),
        "exercise_ratio": _clean(row.exercise_ratio),
        "maturity": _clean(row.expiry),
        "best": {
            "bid": _clean(row.bid), "ask": _clean(row.ask),
            "bid_size": _clean(row.bid_size), "ask_size": _clean(row.ask_size),
        },
    }


def _fold_option_tick(row):
    return {
        "code": row.code,
        "name": _clean(row.name) or row.code,
        "type": _clean(row.type),
        "strike": _clean(row.strike),
        "expiry": _clean(row.expiry),
        "best": {
            "bid": _clean(row.bid), "ask": _clean(row.ask),
            "bid_size": _clean(row.bid_size), "ask_size": _clean(row.ask_size),
        },
    }


def _build_frames(warrant_books, option_books, as_of_date):
    """Flatten the current book caches into the DataFrames
    `static_arb._scan_chain` expects, as of one tick's date."""
    w_rows = []
    for code, w in warrant_books.items():
        maturity = w.get("maturity")
        if maturity is None:
            continue
        best = w["best"]
        w_rows.append({
            "warrant_code": code, "warrant_name": w["name"], "type": w["type"],
            "strike": w["strike"], "exercise_ratio": w["exercise_ratio"],
            "days_to_expiry": (maturity - as_of_date).days,
            "ask": best["ask"], "ask_qty": best["ask_size"],
        })

    o_rows = []
    for code, o in option_books.items():
        expiry = o.get("expiry")
        if expiry is None:
            continue
        best = o["best"]
        bid, ask = best["bid"], best["ask"]
        dte = (expiry - as_of_date).days
        if dte < 1:
            continue  # mirrors options_logic.read_tw_option(min_days=1)
        o_rows.append({
            "contract": code, "type": o["type"], "strike": o["strike"],
            "days_to_expiry": dte,
            "bid": bid, "bid_live": bid is not None and bid > 0, "bid_size": best["bid_size"],
            "ask": ask, "ask_live": ask is not None and ask > 0, "ask_size": best["ask_size"],
        })

    return pd.DataFrame(w_rows), pd.DataFrame(o_rows)


def _solve_book(warrant_df, opt_df, horizons, M, r, min_edge):
    """Every horizon of one tick's book, through whichever engine is live.

    `_scan_chain` builds the legs and solves all the horizons inside the Rust
    kernel; the loop underneath is the scipy path `match_static_arb` falls back
    to, kept here so a machine without the extension still replays (slowly).
    """
    if USE_RUST:
        return static_arb._scan_chain(warrant_df, opt_df, horizons, M, r, min_edge)[0]
    rows = []
    for T_star in horizons:
        longs, shorts, _dropped = static_arb._build_legs(warrant_df, opt_df, T_star, M, r)
        hit = static_arb._solve_horizon(longs, shorts, T_star, min_edge)
        if hit:
            rows.append(hit)
    return rows


def replay_lp(df, stride=1, min_edge=0.0, max_horizon_dte=None,
              M=2000.0, r=None, progress_every=5000):
    r = options_logic.R if r is None else r
    warrant_books, option_books = {}, {}
    out = []
    n = len(df)
    for i, row in enumerate(df.itertuples(index=False)):
        if row.kind == "warrant":
            warrant_books[row.code] = _fold_warrant_tick(row)
        else:
            option_books[row.code] = _fold_option_tick(row)

        if stride > 1 and (i + 1) % stride != 0:
            continue

        warrant_df, opt_df = _build_frames(warrant_books, option_books, row.ts.date())
        if warrant_df.empty or opt_df.empty:
            if progress_every and (i + 1) % progress_every == 0:
                print(f"{i + 1:,}/{n:,} ticks replayed, {len(out):,} hit-rows so far")
            continue

        horizons = sorted(set(int(d) for d in opt_df["days_to_expiry"].unique()))
        if max_horizon_dte:
            horizons = [h for h in horizons if h <= max_horizon_dte]

        for hit in _solve_book(warrant_df, opt_df, horizons, M, r, min_edge):
            hit["underlying_code"] = None
            hit["underlying_price"] = None
            out.append({"tick_ts": row.ts, "tick_kind": row.kind, "tick_code": row.code, **hit})

        if progress_every and (i + 1) % progress_every == 0:
            print(f"{i + 1:,}/{n:,} ticks replayed, {len(out):,} hit-rows so far")

    return pd.DataFrame(out)

In [ ]:
STRIDE = 1             # re-solve the LP after every single tick; raise this only if
                       # you want a faster, coarser first pass instead
MIN_EDGE = 0.0         # NT$ guaranteed-profit floor, same default as match_static_arb
MAX_HORIZON_DTE = None # e.g. 60 to skip far-dated horizons and speed things up

hits_df = replay_lp(raw, stride=STRIDE, min_edge=MIN_EDGE, max_horizon_dte=MAX_HORIZON_DTE)
print(f"\n{len(hits_df):,} (tick, horizon) hit-rows across {raw['ts'].nunique():,} distinct tick timestamps")
hits_df.head()

## Quick summary

`hits_df` has one row per (tick, horizon) structure that survived lot
rounding — the same fields `_solve_horizon` returns (`horizon_dte`,
`n_long`/`n_short`, `legs`, `net_credit`, `min_payoff`, `guaranteed_profit`,
`gross_debit`, `return_pct`, ...) plus which tick triggered it. `legs` is a
list of per-leg dicts (side, instrument, strike, dte, lots, cash) — exactly
what the Arb Finder modal would show for that row.

In [ ]:
if len(hits_df):
    by_horizon = (hits_df.groupby("horizon_dte")
                  .agg(n_ticks=("tick_ts", "count"),
                       max_guaranteed=("guaranteed_profit", "max"),
                       max_net_credit=("net_credit", "max"),
                       first_seen=("tick_ts", "min"),
                       last_seen=("tick_ts", "max"))
                  .sort_values("max_guaranteed", ascending=False))
    display(by_horizon)

    best = hits_df.loc[hits_df["guaranteed_profit"].idxmax()]
    print(f"\nBest structure: horizon={best['horizon_dte']}d, "
          f"{best['n_long']} long leg(s) + {best['n_short']} short leg(s), "
          f"guaranteed_profit={best['guaranteed_profit']:,.0f}, "
          f"net_credit={best['net_credit']:,.0f}, return={best['return_pct']}%")
    for leg in best["legs"]:
        print(f"  {leg['side']:>5} {leg['kind']:<7} {leg['code']} {leg['type']} "
              f"K={leg['strike']} dte={leg['dte']} lots={leg['lots']}{leg['lot_label']} "
              f"cash={leg['cash']:,.0f}")
else:
    print("No static-arb structures found across the whole replay — check the "
          "exercise_ratio and zero-option-ticks warnings above first.")

## Save

Writes the full per-tick hit table next to the source CSV for further
analysis outside this notebook. `legs` is JSON-encoded first since a plain
CSV can't hold a nested list column.

In [ ]:
import json

save_df = hits_df.copy()
if "legs" in save_df.columns:
    save_df["legs"] = save_df["legs"].apply(json.dumps)

out_path = CSV_PATH.with_name(CSV_PATH.stem + "_lp_hits.csv")
save_df.to_csv(out_path, index=False)
print("Saved:", out_path)